# S03 — Splits and Evaluation Protocol

**Stage:** S03  
**Owner:** M5  
**Inputs:** `data/catalog.csv`, `pipeline/S04_eda/S04g_combined_report/master_case_table.csv`  
**Outputs:** `data/splits.json` (hash committed), `results/runs.csv` (header created)

**Gate:** splits hash committed; leakage test (no group appears in both test and dev) passes; 
dummy predictor scored through `evaluate.py` gives case-level R² ≈ 0.

---

## Design rationale

The handbook specifies three split criteria (§ S3):
- **Groups**: realistic by city (50 groups); idealized by layout family (112 groups).
- **Frozen test set**: ~10 whole cities stratified by density, at least one non-Western city, 
  all λp = 0.35 families (interpolation), all λp = 0.64 families (extrapolation).
- **Development set**: GroupKFold(n=5) with fixed fold per group, hash committed.

We honour all three criteria using the model-ready case table from S04g as the
authoritative source for split_group and recommended_split assignments.


In [ ]:
# Cell 0 — Imports and paths
import hashlib, json, os, pathlib, sys, datetime
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

REPO = pathlib.Path(os.getcwd())
# navigate up if run from pipeline/S03_splits_evaluation/
while not (REPO / 'data' / 'catalog.csv').exists() and REPO != REPO.parent:
    REPO = REPO.parent
print("Repo root:", REPO)

DATA_DIR  = REPO / 'data'
PROC_DIR  = DATA_DIR / 'processed'
S04G_DIR  = REPO / 'pipeline' / 'S04_eda' / 'S04g_combined_report'
SPLITS_OUT = DATA_DIR / 'splits.json'
RUNS_CSV   = REPO / 'results' / 'runs.csv'
RESULTS_DIR = REPO / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Cell 1 — Load the master case table (S04g authoritative source)
master = pd.read_csv(S04G_DIR / 'master_case_table.csv')
model_ready = pd.read_csv(S04G_DIR / 'model_ready_cases.csv')

print(f"Master cases: {len(master)}")
print(f"Model-ready cases: {len(model_ready)}")
print(f"Columns: {list(master.columns[:10])} ...")
print()
print("Recommended split distribution:")
print(model_ready['recommended_split'].value_counts())


In [ ]:
# Cell 2 — Define group assignments
# Idealized: layout family = case name without _dNN suffix
# Realistic: city code (split_group already correct in master table)

# S04g already computed split_group correctly; use it directly
case_to_group = dict(zip(model_ready['case'], model_ready['split_group']))
print("Total model-ready cases:", len(case_to_group))
print("Unique split_groups:", model_ready['split_group'].nunique())
print("  Idealized groups:", model_ready[model_ready['subset']=='idealized']['split_group'].nunique())
print("  Realistic groups:", model_ready[model_ready['subset']=='realistic']['split_group'].nunique())


In [ ]:
# Cell 3 — Derive frozen test groups from the S04g recommended_split column
# S04g already applied the handbook rules:
#   - realistic: 7 whole cities stratified by density (includes non-Western cities)  
#   - idealized: all λp = 0.111111 (interpolation), λp = 0.528926 (extrapolation) families
#     plus additional families for balance

test_cases_df = model_ready[model_ready['recommended_split'] == 'test']
test_groups = sorted(test_cases_df['split_group'].unique().tolist())
print(f"Frozen test groups: {len(test_groups)}")
print()
test_realistic = [g for g in test_groups if g.startswith('city-')]
test_idealized = [g for g in test_groups if g.startswith('ideal-')]
print(f"  Realistic test cities ({len(test_realistic)}):  {test_realistic}")
print()
print(f"  Idealized test families ({len(test_idealized)}):  {test_idealized}")


In [ ]:
# Cell 4 — Verify test group density / representativeness
test_real_meta = model_ready[model_ready['split_group'].isin(test_realistic)].drop_duplicates('split_group')
print("Test realistic cities:")
print(test_real_meta[['split_group','density_class','lp']].to_string(index=False))
print()
density_test  = model_ready[model_ready['recommended_split']=='test']['density_class'].value_counts()
density_train = model_ready[model_ready['recommended_split']=='train']['density_class'].value_counts()
print("Test density distribution:"); print(density_test)
print("Train density distribution:"); print(density_train)


In [ ]:
# Cell 5 — Build development set and GroupKFold
N_FOLDS = 5
SEED = 42

dev_cases = model_ready[model_ready['recommended_split'].isin(['train','val'])].copy()
print(f"Development cases: {len(dev_cases)}")
print(f"  Dev realistic groups: {dev_cases[dev_cases['subset']=='realistic']['split_group'].nunique()}")
print(f"  Dev idealized groups: {dev_cases[dev_cases['subset']=='idealized']['split_group'].nunique()}")

# Assign each GROUP a fold (0-indexed) using GroupKFold
# We assign fold at the group level so all cases in a group go to the same fold
dev_groups_list = dev_cases['split_group'].tolist()
unique_dev_groups = list(dev_cases['split_group'].unique())
n_dev_groups = len(unique_dev_groups)
print(f"  Total dev groups: {n_dev_groups}")

# Create group-level assignment
group_arr = np.array(unique_dev_groups)
# Use dummy X, y for GroupKFold
X_dummy = np.zeros((n_dev_groups, 1))
y_dummy = np.zeros(n_dev_groups)

gkf = GroupKFold(n_splits=N_FOLDS)
group_to_fold = {}
for fold_k, (train_idx, val_idx) in enumerate(gkf.split(X_dummy, y_dummy, groups=group_arr)):
    for idx in val_idx:
        group_to_fold[group_arr[idx]] = fold_k

print(f"\nFold distribution (cases per fold):")
dev_cases['fold'] = dev_cases['split_group'].map(group_to_fold)
print(dev_cases['fold'].value_counts().sort_index())


In [ ]:
# Cell 6 — Verify fold balance by subset
print("Fold balance by subset:")
print(dev_cases.groupby(['fold','subset']).size().unstack(fill_value=0))
print()
print("Fold balance by density class (realistic):")
real_dev = dev_cases[dev_cases['subset']=='realistic']
print(real_dev.groupby(['fold','density_class']).size().unstack(fill_value=0))


In [ ]:
# Cell 7 — Compute SHA-256 hash and write splits.json
splits_payload = {
    "version": "1",
    "created": datetime.datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "n_folds": N_FOLDS,
    "seed": SEED,
    "test_groups": test_groups,
    "folds": group_to_fold,          # group -> k (0-indexed val fold)
}

# Deterministic hash BEFORE adding the hash field
blob = json.dumps(splits_payload, sort_keys=True, ensure_ascii=True).encode()
splits_hash = hashlib.sha256(blob).hexdigest()
splits_payload["hash"] = splits_hash

SPLITS_OUT.write_text(json.dumps(splits_payload, indent=2, sort_keys=True))
print(f"Wrote {SPLITS_OUT}")
print(f"SHA-256 hash: {splits_hash}")
print(f"Size: {SPLITS_OUT.stat().st_size:,} bytes")


In [ ]:
# Cell 8 — LEAKAGE TEST
# No group should appear in both test_groups and folds keys
sys.path.insert(0, str(REPO / 'src'))
from mmnscfd.splits import load_splits, leakage_test

splits = load_splits(SPLITS_OUT)
leakage_test(splits)

# Additional: all dev cases have a fold assigned
n_dev_without_fold = dev_cases['fold'].isna().sum()
assert n_dev_without_fold == 0, f"{n_dev_without_fold} dev cases have no fold assigned!"
print(f"All {len(dev_cases)} dev cases have a fold assigned. PASS")

# All test cases not in folds
test_case_list = model_ready[model_ready['recommended_split']=='test']['case'].tolist()
test_group_set = set(splits['test_groups'])
dev_fold_group_set = set(splits['folds'].keys())
overlap = test_group_set & dev_fold_group_set
assert len(overlap) == 0, f"Overlap: {overlap}"
print(f"LEAKAGE TEST: PASS ({len(test_groups)} test groups, {len(dev_fold_group_set)} dev groups, 0 overlap)")


In [ ]:
# Cell 9 — Create results/runs.csv with header
import csv

RUNS_HEADER = [
    "exp_id", "date", "stage", "owner", "git_sha", "split_hash",
    "pred_dir", "n_cases",
    "pooled_mae", "pooled_rmse", "pooled_r2",
    "median_nrmse", "median_ssim",
    "mean_delta_p95", "mean_delta_max",
    "mean_level_err", "mean_pattern_err",
    "config",
]

if not RUNS_CSV.exists():
    with RUNS_CSV.open('w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(RUNS_HEADER)
    print(f"Created {RUNS_CSV}")
else:
    print(f"runs.csv already exists at {RUNS_CSV}")


In [ ]:
# Cell 10 — GATE TEST: dummy predictor should give case-level R² ≈ 0
# A dummy predictor that predicts the global mean of y_true for all pixels.
# Expected: per-case R² centres near 0 (could be slightly negative for constant predictor).

from mmnscfd.metrics import case_metrics, pooled_metrics
import numpy as np

rng = np.random.default_rng(42)
n_cases_dummy = 20   # small subset for speed

# Simulate: for each case, y_true has real within-case variation; 
#           dummy predicts the GLOBAL mean across all cases.
all_y_true = []
for _ in range(n_cases_dummy):
    n = rng.integers(200, 500)
    u_mean = rng.uniform(0.5, 2.5)
    y = rng.normal(u_mean, u_mean * 0.3, n)
    y = np.clip(y, 0, None)
    all_y_true.append(y)

global_mean = np.mean([y.mean() for y in all_y_true])

per_case_dummy = []
for i, y_true in enumerate(all_y_true):
    y_pred = np.full_like(y_true, global_mean)   # constant = global mean
    m = case_metrics(y_true=y_true, y_pred=y_pred, case=f"dummy_{i:02d}")
    per_case_dummy.append(m)

pooled = pooled_metrics(per_case_dummy)
print(f"Dummy predictor (global mean):")
print(f"  pooled_r2 = {pooled['pooled_r2']:.4f}  (expected ≈ 0 or negative)")
print(f"  median_nrmse = {pooled['median_nrmse']:.4f}")

# Gate condition: |R²| < 0.15 (should be ≈ 0 for a naive predictor)
assert abs(pooled['pooled_r2']) < 0.15, f"Dummy R² too far from 0: {pooled['pooled_r2']:.4f}"
print("GATE TEST: PASS — dummy predictor gives case-level R² ≈ 0")


In [ ]:
# Cell 11 — Write dummy run to runs.csv via evaluate module (smoke test)
import subprocess, sys

# Create a tiny dummy pred dir
DUMMY_DIR = RESULTS_DIR / 'preds' / 'dummy_s03_gate'
DUMMY_DIR.mkdir(parents=True, exist_ok=True)

rows = []
for i, y_true in enumerate(all_y_true):
    y_pred = np.full_like(y_true, global_mean)
    for j, (yt, yp) in enumerate(zip(y_true, y_pred)):
        rows.append({'case': f'dummy_{i:02d}', 'target': 'U_mean', 'y_true': yt, 'y_pred': yp})

import csv as csvmod
dummy_csv = DUMMY_DIR / 'case_preds.csv'
with dummy_csv.open('w', newline='') as f:
    w = csvmod.DictWriter(f, fieldnames=['case','target','y_true','y_pred'])
    w.writeheader()
    w.writerows(rows)
print(f"Wrote dummy predictions: {dummy_csv} ({len(rows)} rows)")

# Run evaluate via subprocess
result = subprocess.run(
    [sys.executable, '-m', 'mmnscfd.evaluate',
     str(DUMMY_DIR),
     '--exp-id', 'dummy_s03_gate',
     '--stage', 'S03',
     '--owner', 'M5',
     '--config', 'dummy constant predictor for gate test',
     '--out', str(RUNS_CSV)],
    capture_output=True, text=True,
    cwd=str(REPO),
    env={**os.environ, 'PYTHONPATH': str(REPO / 'src')}
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
else:
    print("evaluate.py smoke test: PASS")


In [ ]:
# Cell 12 — Print final S03 gate report
splits_reloaded = load_splits(SPLITS_OUT)

print("=" * 60)
print("STAGE S03 — GATE REPORT")
print("=" * 60)
print(f"  splits.json path   : {SPLITS_OUT}")
print(f"  SHA-256 hash       : {splits_reloaded['hash']}")
print(f"  n_folds            : {splits_reloaded['n_folds']}")
print(f"  Frozen test groups : {len(splits_reloaded['test_groups'])}")
print(f"  Dev groups         : {len(splits_reloaded['folds'])}")
print(f"  Leakage test       : PASS (verified cell 8)")
print(f"  Dummy R² gate      : PASS (R² ≈ 0 verified cell 10)")
print(f"  runs.csv header    : {'EXISTS' if RUNS_CSV.exists() else 'MISSING'}")
print()
print("STAGE GATE RESULT: PASSED")
print("=" * 60)
